In [3]:

import sys
if 'google.colab' in sys.modules:
    !pip install utdquake
    
lib = None
lib = "/home/emmanuel/ecastillo/dev/utdquake"
if lib is not None:
    sys.path.append(lib)

# Version (Required)

In [4]:
import utdquake
from packaging.version import Version

current_version = Version(utdquake.__version__)
required_version = Version("0.0.10")

if current_version < required_version:
    raise Exception(f"Please update utdquake to at least version {required_version}")

# UTDClient

## get custom stations

In [7]:
from utdquake.clients.fdsn.client import Client

provider = "IRIS"
out = "./custom_events"

client =  Client(provider)
sta = client.get_custom_stations(output_folder=out,network="TX",station="PB*")
print(sta.head(10))

     sta_id network station   latitude   longitude  elevation  \
0   TX.PB01      TX    PB01  30.943670 -103.781120     1010.0   
1   TX.PB02      TX    PB02  31.408951 -103.510162      792.0   
3   TX.PB03      TX    PB03  31.083840 -103.513950      871.0   
4   TX.PB04      TX    PB04  31.186970 -103.269400      812.0   
5   TX.PB05      TX    PB05  30.919780 -103.324700      957.0   
6   TX.PB06      TX    PB06  31.647200 -103.218250      831.0   
9   TX.PB07      TX    PB07  31.579350 -103.667930      856.0   
10  TX.PB08      TX    PB08  30.891740 -102.907360      926.0   
11  TX.PB09      TX    PB09  31.774145 -104.301444     1139.0   
12  TX.PB10      TX    PB10  31.283607 -103.754585      858.0   

             starttime endtime                       site_name  
0  2017-09-13 00:00:00     NaT            Balmorhea State Park  
1  2017-01-27 00:00:00     NaT  Crockett Middle School (Pecos)  
3  2021-07-28 03:30:00     NaT                        Verhalen  
4  2019-02-12 00:00:00  

## get custom events

In [8]:
from obspy import UTCDateTime
from utdquake.clients.fdsn.client import Client

region = [-104.84329,-103.79942,31.39610,31.91505] 
provider = "USGS"
out = "./custom_events"

client =  Client(provider)
cat,picks,mag = client.get_custom_events(starttime=UTCDateTime("2024-04-18T23:00:00"),
                        endtime=UTCDateTime("2024-04-19T23:00:00"),
                        minlatitude=region[2], maxlatitude=region[3], 
                        minlongitude=region[0], maxlongitude=region[1],
                        debug=True,
                        output_folder=out,
                        )


Event id 1/7: tx2024hrey
Event id 2/7: tx2024hrgy
Event id 3/7: tx2024hriz
Event id 4/7: tx2024hrlf
Event id 5/7: tx2024hrwl
Event id 6/7: tx2024hrwo
Event id 7/7: tx2024hstr


# How to read?

FIles in your ouput folder

In [10]:
import os
print(os.listdir(out))

['mags.db', 'stations.csv', 'picks.db', 'origin.csv']


picks and mags are in database format 

In [13]:
import os

stations_path = os.path.join(out,"stations.csv")
catalog_path = os.path.join(out,"origin.csv")
picks_path = os.path.join(out,"picks.db")
mags_path = os.path.join(out,"mags.db")

catalog and stations

In [14]:
import pandas as pd

catalog = pd.read_csv(catalog_path)
stations = pd.read_csv(stations_path)

print(catalog.head())
print(stations.head())

        ev_id     ev_type agency                 origin_time  longitude  \
0  tx2024hrey  earthquake     TX  2024-04-19 00:33:29.083274   -103.884   
1  tx2024hrgy  earthquake     TX  2024-04-19 01:33:51.844213   -104.194   
2  tx2024hriz  earthquake     TX  2024-04-19 02:34:59.666935   -103.937   
3  tx2024hrlf  earthquake     TX  2024-04-19 03:43:00.012014   -104.287   
4  tx2024hrwl  earthquake     TX  2024-04-19 09:23:13.067845   -104.410   

   latitude   depth loc_method_id        earth_model_id  magnitude  ...  \
0    31.488  6074.2     NonLinLoc  PB1D-20170918-topoLV        1.8  ...   
1    31.612  4920.7     NonLinLoc  PB1D-20170918-topoLV        1.8  ...   
2    31.605  5920.4     NonLinLoc  PB1D-20170918-topoLV        1.6  ...   
3    31.717  6151.1     NonLinLoc  PB1D-20170918-topoLV        2.6  ...   
4    31.677  5638.4     NonLinLoc  PB1D-20170918-topoLV        2.9  ...   

  qc_magnitude_evaluation_status qc_associated_phase_count  \
0                      confirmed    

picks and mags (all)

In [15]:
from utdquake.core.database.database import load_from_sqlite

picks = load_from_sqlite(picks_path)
mags = load_from_sqlite(mags_path)

print(picks.head())
print(mags.head())

        ev_id network station location channel phase_hint  \
0  tx2024hrey      TX    PECS       00     HH2          P   
1  tx2024hrey      TX    PECS       00     HH2          S   
2  tx2024hrey      TX    PB40       00     HH1          P   
3  tx2024hrey      TX    PB40       00     HH2          S   
4  tx2024hrey      TX    PB43       00     HH1          P   

                         time  time_lower_error  time_upper_error   author  \
0  2024-04-19 00:33:32.130765               0.2               0.2  Anestis   
1  2024-04-19 00:33:34.698990               0.3               0.3  Anestis   
2  2024-04-19 00:33:32.888062               0.2               0.2  Anestis   
3  2024-04-19 00:33:36.081880               0.2               0.2  Anestis   
4  2024-04-19 00:33:33.497192               0.1               0.1  Anestis   

   ... polarity evaluation_mode evaluation_status time_correction     azimuth  \
0  ...     None          manual              None             0.0  173.168308   
1 

by chunks

In [19]:
from utdquake.core.database.database import load_chunks_from_sqlite

picks_chunks = load_chunks_from_sqlite(picks_path,chunksize=2)
mags_chunks = load_chunks_from_sqlite(mags_path,chunksize=2)

for picks in picks_chunks:
    print(picks)
    
for mags in mags_chunks:
    print(mags)

<generator object load_chunks_from_sqlite at 0x75bbebffb140>
          ev_id network station location channel phase_hint  \
0    tx2024hrey      TX    PECS       00     HH2          P   
1    tx2024hrey      TX    PECS       00     HH2          S   
2    tx2024hrey      TX    PB40       00     HH1          P   
3    tx2024hrey      TX    PB40       00     HH2          S   
4    tx2024hrey      TX    PB43       00     HH1          P   
..          ...     ...     ...      ...     ...        ...   
106  tx2024hrgy      TX    PB12       00     HHZ          P   
107  tx2024hrgy      GM   NMP11       01     HHZ          P   
108  tx2024hrgy      US    MNTX       00     BHZ          P   
109  tx2024hrgy      TX    PB46       00     HHZ          P   
110  tx2024hrgy      TX    PB01       00     HHZ          P   

                           time  time_lower_error  time_upper_error   author  \
0    2024-04-19 00:33:32.130765               0.2               0.2  Anestis   
1    2024-04-19 00:33: